# CourtVisionNet Training on Google Colab

Train the badminton court keypoint detection model on a GPU.

**Prerequisites:**
1. Upload `colab_data.zip` to your Google Drive root (or adjust the path below)
2. Set Runtime > Change runtime type > **T4 GPU** (or better)
3. Run all cells in order

## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === CONFIGURE THESE ===
DRIVE_ZIP_PATH = '/content/drive/MyDrive/colab_data.zip'  # Path to your zip in Drive
REPO_URL = 'https://github.com/roasteduck04/badminton-court-finder.git'
BRANCH = 'main'

In [ ]:
# Clone the repo
import os
if not os.path.exists('/content/badminton-court-finder'):
    !git clone --branch {BRANCH} {REPO_URL} /content/badminton-court-finder
else:
    !cd /content/badminton-court-finder && git pull

%cd /content/badminton-court-finder

In [ ]:
# Install dependencies (torch/torchvision already available on Colab)
!pip install -q albumentations>=2.0.0 shapely>=2.0

In [ ]:
# Unzip data
import zipfile
DATA_DIR = '/content/badminton-court-finder/data/colab_data'

if not os.path.exists(DATA_DIR):
    print(f'Unzipping {DRIVE_ZIP_PATH} ...')
    with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zf:
        zf.extractall('/content/badminton-court-finder/data')
    print('Done.')
else:
    print('Data already unzipped.')

# Verify
for split in ['train', 'valid', 'test']:
    ann_dir = os.path.join(DATA_DIR, split, 'annotations')
    img_dir = os.path.join(DATA_DIR, split, 'images')
    n_ann = len([f for f in os.listdir(ann_dir) if f.endswith('.json')])
    n_img = len(os.listdir(img_dir))
    print(f'  {split}: {n_ann} annotations, {n_img} images')

## 2. Verify GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

## 3. Train

In [ ]:
from src.training.config import TrainConfig
from src.training.train import train

config = TrainConfig(
    # Data paths
    train_annotations=os.path.join(DATA_DIR, 'train', 'annotations'),
    val_annotations=os.path.join(DATA_DIR, 'valid', 'annotations'),
    train_images=os.path.join(DATA_DIR, 'train', 'images'),
    val_images=os.path.join(DATA_DIR, 'valid', 'images'),

    # Model
    image_size=640,
    heatmap_size=160,
    pretrained=True,

    # Training
    batch_size=8,        # Increase to 16 if VRAM allows (e.g. A100)
    num_epochs=100,
    learning_rate=3e-5,
    weight_decay=1e-4,
    patience=15,
    freeze_backbone_epochs=5,

    # Colab: use multiple workers for faster data loading
    num_workers=2,

    # Save checkpoints to Drive so they persist
    checkpoint_dir='/content/drive/MyDrive/cvn_checkpoints',
)

print(f'Train: {len(os.listdir(config.train_annotations))} annotations')
print(f'Valid: {len(os.listdir(config.val_annotations))} annotations')
print(f'Checkpoints: {config.checkpoint_dir}')

In [ ]:
# Train!
results = train(config)
print(f"\nTraining complete!")
print(f"Best validation loss: {results['best_val_loss']:.4f}")
print(f"Final epoch: {results['final_epoch']}")

## 4. Evaluate

In [ ]:
from src.models.courtvisionnet import CourtVisionNet
from src.models.losses import CourtVisionLoss
from src.training.dataset import CourtDataset
from src.training.train import validate
from src.preprocessing.augmentation import get_val_transforms
from torch.utils.data import DataLoader

# Load best model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CourtVisionNet(
    in_channels=7, num_keypoints=30,
    image_size=640, heatmap_size=160, pretrained=False
).to(device)

ckpt = torch.load(os.path.join(config.checkpoint_dir, 'best_model.pt'), map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded best model from epoch {ckpt['epoch']}")

# Evaluate on test set
test_ds = CourtDataset(
    os.path.join(DATA_DIR, 'test', 'annotations'),
    os.path.join(DATA_DIR, 'test', 'images'),
    transform=get_val_transforms(640),
    image_size=640, heatmap_size=160,
)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)

loss_fn = CourtVisionLoss()
test_loss, test_components, test_metrics = validate(model, test_loader, loss_fn, device)

print(f"\nTest Results:")
print(f"  Loss: {test_loss:.4f}")
for k, v in test_components.items():
    print(f"  {k}: {v:.4f}")
print(f"  PCK@10: {test_metrics['pck_at_10']:.4f}")
print(f"  MRE: {test_metrics['mre']:.2f} px")

## 5. Training History

In [ ]:
import matplotlib.pyplot as plt

history = results['history']
epochs = [h['epoch'] for h in history]
train_losses = [h['train_loss'] for h in history]
val_losses = [h['val_loss'] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, train_losses, label='Train')
ax1.plot(epochs, val_losses, label='Validation')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot individual loss components
component_keys = [k for k in history[0] if k.startswith('val_') and k != 'val_loss']
for k in component_keys:
    ax2.plot(epochs, [h[k] for h in history], label=k.replace('val_', ''))
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Validation Loss Components')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config.checkpoint_dir, 'training_history.png'), dpi=150)
plt.show()

## 6. Download Checkpoint

The best model is saved to Google Drive at the `checkpoint_dir` path above. You can also download it directly:

In [ ]:
from google.colab import files
best_model_path = os.path.join(config.checkpoint_dir, 'best_model.pt')
print(f'Best model: {best_model_path}')
print(f'Size: {os.path.getsize(best_model_path) / 1024**2:.1f} MB')
# Uncomment to download:
# files.download(best_model_path)